# Beyond the Headline Rate
## Business Analytics Case Study

**Goal:** quantify the difference between advertised personal-loan rates and effective annualized borrowing cost using public Indian lender terms.

**Data note:** the 10k+ rows are standardized loan-amount × tenure scenarios generated from public lender terms. They are not 10k independent lender advertisements.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf

df = pd.read_csv("data/scenario_dataset_13860.csv")
print(df.shape)
display(df.head())

## 1. Data quality checks

In [ ]:
print(df.isna().sum().sort_values(ascending=False).head(15))
print("Duplicate rows:", df.duplicated().sum())
print("Lenders:", df.lender_name.nunique())
print(df.lender_type.value_counts())

## 2. Descriptive statistics

In [ ]:
display(df[[
    "advertised_rate_nominal_pct","effective_apr_pct",
    "transparency_gap_pp","fee_induced_gap_pp",
    "processing_fee_inr","emi_inr","total_repayment_inr"
]].describe().T)

## 3. Visualizations

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(df["transparency_gap_pp"], bins=50)
plt.xlabel("Effective APR - advertised rate (percentage points)")
plt.ylabel("Scenario count")
plt.title("Distribution of transparency gap")
plt.show()

plt.figure(figsize=(9,5))
df.groupby("lender_type")["transparency_gap_pp"].median().sort_values().plot(kind="bar")
plt.ylabel("Median transparency gap (pp)")
plt.title("Median transparency gap by lender type")
plt.xticks(rotation=0)
plt.show()

## 4. Lender-level transparency score

This is a study-specific analytical proxy, not a regulatory or legal judgement.

In [ ]:
summary = df.groupby(["lender_name","lender_type"]).agg(
    median_gap_pp=("transparency_gap_pp","median"),
    mean_gap_pp=("transparency_gap_pp","mean"),
    std_gap_pp=("transparency_gap_pp","std"),
    median_apr_pct=("effective_apr_pct","median"),
    median_fee_gap_pp=("fee_induced_gap_pp","median"),
    n_scenarios=("lender_name","size")
).reset_index()

summary["gap_rank_pct"] = summary["median_gap_pp"].rank(pct=True)
summary["variability_rank_pct"] = summary["std_gap_pp"].rank(pct=True)
summary["transparency_score"] = 100 * (
    1 - (0.80*summary["gap_rank_pct"] + 0.20*summary["variability_rank_pct"])
)
summary["tier"] = pd.qcut(
    summary["transparency_score"], 3,
    labels=["Highly Misleading","Moderately Misleading","Transparent"]
)
display(summary.sort_values("transparency_score", ascending=False))

## 5. Group comparison

In [ ]:
groups = [g["transparency_gap_pp"].dropna().values for _, g in df.groupby("lender_type")]
print(stats.kruskal(*groups))

## 6. Regression

The regression is explanatory, not causal. It estimates the association between the gap and rate, fee burden, loan amount, tenure and lender type.

In [ ]:
reg = df.copy()
reg["loan_amount_lakh"] = reg["loan_amount_inr"]/100000
reg["fee_pct_of_loan"] = reg["processing_fee_inr"]/reg["loan_amount_inr"]*100

model = smf.ols(
    "transparency_gap_pp ~ advertised_rate_nominal_pct + fee_pct_of_loan + tenure_months + loan_amount_lakh + C(lender_type)",
    data=reg
).fit()
print(model.summary())

## 7. Sensitivity analysis

Because many source fees are published as "up to", the current scenario dataset uses the published upper bound. Do not describe this as the fee every borrower pays. Re-run with exact/lower/base/upper assumptions when source data allows.

In [ ]:
cutoff = df["effective_apr_pct"].quantile(0.99)
robust = df[df["effective_apr_pct"] <= cutoff]
print("99th percentile APR cutoff:", cutoff)
print(robust["transparency_gap_pp"].describe())

In [ ]:
summary.to_csv("data/lender_transparency_summary.csv", index=False)
print("Saved lender summary.")